# Nordeus Challenge Data Science

* Predicting Clan Tournament Winners

The goal of this project is to predict which clan will win a weekend tournament match in Top Eleven. Each clan has 6 managers who compete against 6 managers from a rival clan. I used member-level statistics to build clan-level features and trained a machine learning model to predict the winner.

In [20]:
import pandas as pd
import numpy as np

matches_train = pd.read_csv('clan_matches_training.csv')
matches_test  = pd.read_csv('clan_matches_test.csv')
stats_train   = pd.read_csv('member_stats_training.csv')
stats_test    = pd.read_csv('member_stats_test.csv')

print("TRAIN MATCHES")
print(matches_train.shape)
print(matches_train.head(3))
print("\nTarget:")
print(matches_train['clan_winner'].value_counts())

print("\nMEMBER STATS")
print(stats_train.shape)
print(stats_train.head(3))
print("\nMissing values:")
print(stats_train.isnull().sum())

TRAIN MATCHES
(24288, 5)
    clan_1_id   clan_2_id  clan_1_points  clan_2_points  clan_winner
0   clan_7201  clan_25726              0             72            2
1   clan_5202  clan_16269              0             78            2
2  clan_26609   clan_2868              0             84            2

Target:
clan_winner
2    12162
1    12126
Name: count, dtype: int64

MEMBER STATS
(291456, 15)
  clan_id user_id  days_active_last_28_days  days_active_last_7_days  \
0  clan_1  user_1                        28                        7   
1  clan_2  user_2                        27                        7   
2  clan_3  user_3                        28                        7   

   days_since_last_active registration_country registration_platform_specific  \
0                       0               Israel                  Android Phone   
1                       0             Malaysia                  Android Phone   
2                       0               Poland                  Android

* Features for Clan Level

Since each clan has 6 members, I aggregated individual player statistics (mean, max, min, std, sum) to get one row per clan. I also created custom features based on game mechanics, such as weighted quality (stars * multiplier) and activity score (active days * training count).

In [ ]:
def build_clan_features(stats_df):
    df = stats_df.copy()
    
    df['is_payer_lifetime'] = df['is_payer_lifetime'].map(
        {'True': 1, 'False': 0, True: 1, False: 0}
    ).fillna(0)
    
    seg_map = {'0) Non-payer': 0, '1) Lapsed': 1, '2) Low': 2, '3) Mid': 3, '4) Whale': 4}
    df['payment_segment_ord'] = df['dynamic_payment_segment'].map(seg_map).fillna(0)
    
    numeric_cols = [
        'days_active_last_28_days', 'days_active_last_7_days',
        'days_since_last_active', 'cohort_day',
        'training_count_last_28_days', 'is_payer_lifetime',
        'payment_segment_ord', 'clan_multiplier',
        'avg_stars_top_11_players', 'avg_stars_top_3_players',
        'avg_training_bonus'
    ]
    
    agg = df.groupby('clan_id')[numeric_cols].agg(['mean', 'max', 'min', 'std', 'sum'])
    agg.columns = ['_'.join(c) for c in agg.columns]
    agg = agg.reset_index()
    
    agg['member_count'] = df.groupby('clan_id')['user_id'].count().values
    
    # stars * clan_multiplier
    df['weighted_quality'] = df['avg_stars_top_11_players'] * df['clan_multiplier']
    wq = df.groupby('clan_id')['weighted_quality'].agg(['sum', 'mean']).reset_index()
    wq.columns = ['clan_id', 'weighted_quality_sum', 'weighted_quality_mean']
    agg = agg.merge(wq, on='clan_id')
    
    #  aktivnost * treninzi
    df['activity_score'] = df['days_active_last_28_days'] * df['training_count_last_28_days']
    act = df.groupby('clan_id')['activity_score'].agg(['sum', 'mean']).reset_index()
    act.columns = ['clan_id', 'activity_score_sum', 'activity_score_mean']
    agg = agg.merge(act, on='clan_id')
    
    # neaktivnih (>3 dana bez login-a)
    df['is_inactive'] = (df['days_since_last_active'] > 3).astype(int)
    inact = df.groupby('clan_id')['is_inactive'].mean().reset_index()
    inact.columns = ['clan_id', 'inactive_frac']
    agg = agg.merge(inact, on='clan_id')
    
    pf = df.groupby('clan_id')['is_payer_lifetime'].mean().reset_index()
    pf.columns = ['clan_id', 'payer_frac']
    agg = agg.merge(pf, on='clan_id')
    
    return agg

clan_train = build_clan_features(stats_train)
clan_test  = build_clan_features(stats_test)

print("Clan train shape:", clan_train.shape)
print("Clan test shape: ", clan_test.shape)

Clan train shape: (48576, 63)
Clan test shape:  (47292, 63)


* Match-Level Features

To compare two clans directly, I merged both clans statistics into a single row per match and calculated the difference and ratio between clan_1 and clan_2 for every feature. A positive diff means clan_1 is stronger in that metric.

In [21]:
def build_match_features(matches_df, clan_feats):
    df = matches_df.copy()
    
    c1 = clan_feats.add_prefix('c1_').rename(columns={'c1_clan_id': 'clan_1_id'})
    c2 = clan_feats.add_prefix('c2_').rename(columns={'c2_clan_id': 'clan_2_id'})
    
    df = df.merge(c1, on='clan_1_id', how='left')
    df = df.merge(c2, on='clan_2_id', how='left')
    
    feat_cols = [c for c in clan_feats.columns if c != 'clan_id']
    
    for col in feat_cols:
        c1c = f'c1_{col}'
        c2c = f'c2_{col}'
        if c1c in df.columns and c2c in df.columns:
            df[f'diff_{col}']  = df[c1c] - df[c2c]
            df[f'ratio_{col}'] = df[c1c] / df[c2c].replace(0, np.nan)
    
    return df

train_df = build_match_features(matches_train, clan_train)
test_df  = build_match_features(matches_test,  clan_test)

drop_cols = ['clan_1_id', 'clan_2_id', 'clan_1_points', 'clan_2_points', 'clan_winner']
feature_cols = [c for c in train_df.columns if c not in drop_cols]

X_train = train_df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
y_train = (train_df['clan_winner'] == 1).astype(int)

X_test  = test_df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)

print(f"Feature matrica: {X_train.shape}")
print(f"Clan_1 win rate u train setu: {y_train.mean():.4f}")

Feature matrica: (24288, 276)
Clan_1 win rate u train setu: 0.4993


* Model Training and Cross-Validation

I trained three models using 5-fold stratified cross-validation to get a reliable estimate of accuracy before predicting on the test set. The models used are Gradient Boosting (GBM), Random Forest (RF) and Logistic Regression (LR).

In [22]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

gbm = GradientBoostingClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=4,
    subsample=0.8, min_samples_leaf=20, random_state=42
)
scores_gbm = cross_val_score(gbm, X_train, y_train, cv=cv, scoring='accuracy')
print(f"GBM  CV accuracy: {scores_gbm.mean():.4f} ± {scores_gbm.std():.4f}")

rf = RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=20,
    random_state=42, n_jobs=-1
)
scores_rf = cross_val_score(rf, X_train, y_train, cv=cv, scoring='accuracy')
print(f"RF   CV accuracy: {scores_rf.mean():.4f} ± {scores_rf.std():.4f}")

lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(C=0.1, max_iter=500, random_state=42))
])
scores_lr = cross_val_score(lr_pipe, X_train, y_train, cv=cv, scoring='accuracy')
print(f"LR   CV accuracy: {scores_lr.mean():.4f} ± {scores_lr.std():.4f}")

GBM  CV accuracy: 0.5812 ± 0.0078
RF   CV accuracy: 0.5826 ± 0.0047
LR   CV accuracy: 0.5797 ± 0.0077


* Final Predictions

After cross-validation, I retrained all three models on the full training set. The final prediction is a weighted ensemble for each model votes on the winner, with weights proportional to their cross-validation accuracy. If the ensemble probability is above 0.5, clan_1 wins, otherwise clan_2 wins.

In [25]:
gbm.fit(X_train, y_train)
rf.fit(X_train, y_train)
lr_pipe.fit(X_train, y_train)

prob_gbm = gbm.predict_proba(X_test)[:, 1]
prob_rf  = rf.predict_proba(X_test)[:, 1]
prob_lr  = lr_pipe.predict_proba(X_test)[:, 1]

w_gbm = scores_gbm.mean()
w_rf  = scores_rf.mean()
w_lr  = scores_lr.mean()
total = w_gbm + w_rf + w_lr

prob_ensemble = (w_gbm * prob_gbm + w_rf * prob_rf + w_lr * prob_lr) / total
y_pred_binary = (prob_ensemble > 0.5).astype(int)
predicted_winner = np.where(y_pred_binary == 1, 1, 2)

print("Raspodela predikcija:")
unique, counts = np.unique(predicted_winner, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  Clan_{u} pobednik: {c} ({c/len(predicted_winner)*100:.1f}%)")

Raspodela predikcija:
  Clan_1 pobednik: 11656 (49.3%)
  Clan_2 pobednik: 11990 (50.7%)


* Submission

In [26]:
submission = pd.DataFrame({
    'clan_1_id': matches_test['clan_1_id'],
    'clan_2_id': matches_test['clan_2_id'],
    'predicted_clan_winner': predicted_winner
})

submission.to_csv('clan_winner_predictions.csv', index=False)
print(submission.head(10))

      clan_1_id     clan_2_id  predicted_clan_winner
0  clan_5029188  clan_5045813                      2
1  clan_5016270  clan_5010361                      1
2  clan_5024341  clan_5015291                      1
3  clan_5025374  clan_5002293                      1
4  clan_5036635  clan_5029684                      1
5  clan_5008910  clan_5033381                      2
6  clan_5021926  clan_5029744                      2
7  clan_5035595  clan_5005752                      1
8  clan_5015113  clan_5027408                      2
9  clan_5014326  clan_5014687                      1


* Feature Importance

The most important feature by far is diff_avg_training_bonus_min, with an importance of 0.44. This means the difference in the weakest member's training bonus between the two clans is the single strongest predictor of match outcome. A clan is only as strong as its weakest link.

In [27]:
import matplotlib.pyplot as plt

feat_imp = pd.Series(gbm.feature_importances_, index=feature_cols)
top20 = feat_imp.sort_values(ascending=False).head(20)

print("\nTop 10:")
print(top20.head(10).to_string())


Top 10:
diff_weakest_training_bonus             0.156029
diff_avg_training_bonus_min             0.108870
ratio_avg_training_bonus_std            0.021730
c1_avg_training_bonus_max               0.015419
c2_avg_training_bonus_max               0.015014
diff_avg_training_bonus_std             0.011798
c2_avg_stars_top_3_players_std          0.011344
diff_training_count_last_28_days_min    0.009333
c2_avg_stars_top_11_players_max         0.009308
c2_cohort_day_std                       0.009103


* Bonus-Clan Advisor

This advisor compares two clans across key metrics and identifies which players need improvement. Players who are weak in multiple areas get a higher priority score, so the clan knows exactly who to focus on first.

In [36]:
def clan_advisor_v2(clan_id, opponent_clan_id, stats_df):
    df = stats_df.copy()
    df['is_payer_lifetime'] = df['is_payer_lifetime'].map(
        {'True': 1, 'False': 0, True: 1, False: 0}).fillna(0)
    
    my_clan  = df[df['clan_id'] == clan_id].copy()
    opp_clan = df[df['clan_id'] == opponent_clan_id].copy()
    
    if my_clan.empty or opp_clan.empty:
        print("Clan is not found.")
        return

    metrics = {
        'Activity (28 dana)':          ('days_active_last_28_days', False),
        'Training Count (28 dana)':      ('training_count_last_28_days', False),
        'Team Quality (top 11 stars)': ('avg_stars_top_11_players', False),
        'Clan Multiplier':              ('clan_multiplier', False),
        'Training Bonus':               ('avg_training_bonus', False),
        'Inactive Members (>3 dana)':  ('days_since_last_active', True),
    }

    print(f"{clan_id}  vs  {opponent_clan_id}")
    print('\n')
    print(f"{'Metric':<38} {'You':>8} {'Opponent':>8}  Status")

    weaknesses = []
    for label, (col, lower_is_better) in metrics.items():
        if col == 'days_since_last_active':
            my_val  = (my_clan[col] > 3).mean()
            opp_val = (opp_clan[col] > 3).mean()
        else:
            my_val  = my_clan[col].mean()
            opp_val = opp_clan[col].mean()

        better = (my_val <= opp_val) if lower_is_better else (my_val >= opp_val)
        status = "better" if better else "worse"
        print(f"{label:<38} {my_val:>8.2f} {opp_val:>8.2f}  {status}")

        if not better:
            gap = abs(my_val - opp_val) / (abs(opp_val) + 1e-6) * 100
            weaknesses.append((label, col, my_val, opp_val, gap, lower_is_better))

    print(f"\n")
    print("Attention to players:")

    if weaknesses:
        player_problem_score = {}
        player_details = {}

        for label, col, my_val, opp_val, gap, lower_is_better in weaknesses:
            if col == 'days_since_last_active':
                worst = my_clan[my_clan[col] > 3][['user_id', col]]
            else:
                threshold = my_clan[col].median()
                worst = my_clan[my_clan[col] < threshold][['user_id', col]]

            for _, row in worst.iterrows():
                uid = row['user_id']
                if uid not in player_problem_score:
                    player_problem_score[uid] = 0
                    player_details[uid] = []
                player_problem_score[uid] += gap
                player_details[uid].append(f"{label}: {row[col]:.1f}")

        sorted_players = sorted(player_problem_score.items(), 
                                key=lambda x: x[1], reverse=True)

        for uid, score in sorted_players[:4]:  
            print(f"{uid}  (priority: {score:.0f})")
            for detail in player_details[uid]:
                print(f"{detail}")

        advice_map = {
            'days_active_last_28_days':    "Log every day.",
            'training_count_last_28_days': "Train more times daily.",
            'avg_stars_top_11_players':    "Improve players in the top 11.",
            'clan_multiplier':             "Better league results = higher multiplier.",
            'avg_training_bonus':          "Invest in training bonuses.",
            'days_since_last_active':      "Motivate inactive members.",
        }
        print('\n')
        print("Plan:")
        weaknesses.sort(key=lambda x: x[4], reverse=True)
        for i, (label, col, my_v, opp_v, gap, _) in enumerate(weaknesses, 1):
            print(f"  {i}. {label}  {gap:.1f}%): {advice_map[col]}")
    else:
        print("All players are in good shape!")


example_clan1 = stats_train['clan_id'].unique()[0]
example_clan2 = stats_train['clan_id'].unique()[1]
clan_advisor_v2(example_clan1, example_clan2, stats_train)

clan_1  vs  clan_2


Metric                                      You Opponent  Status
Activity (28 dana)                        28.00    27.50  better
Training Count (28 dana)                 665.33   214.17  better
Team Quality (top 11 stars)                7.46     7.93  worse
Clan Multiplier                            3.00     2.33  better
Training Bonus                            13.25    14.08  worse
Inactive Members (>3 dana)                 0.00     0.00  better


Attention to players:
user_158343  (priority: 12)
Team Quality (top 11 stars): 6.8
Training Bonus: 10.5
user_183841  (priority: 12)
Team Quality (top 11 stars): 6.4
Training Bonus: 11.5
user_231619  (priority: 12)
Team Quality (top 11 stars): 6.3
Training Bonus: 12.5


Plan:
  1. Training Bonus  5.9%): Invest in training bonuses.
  2. Team Quality (top 11 stars)  5.9%): Improve players in the top 11.
